# Qwen2.5-VL-3B QLoRA train nối tiếp - Chunk 02: 02501–05000

Notebook này train **nối tiếp**. Chunk 1 train từ base model; chunk 2 load adapter chunk 1; chunk 3 load adapter chunk 2; ...

Output chính lưu ở Drive: `/content/drive/MyDrive/Final_Deeplearning/qwen25_vl_herb_qlora_10chunks`.


In [ ]:
# ============================================================
# 1. Cài thư viện
# Không cài lại torch để tránh lệch CUDA trên Colab.
# ============================================================
%pip -q install "transformers==4.51.3" "accelerate==1.6.0" "peft==0.15.2" "bitsandbytes==0.45.5" "qwen-vl-utils" "pillow" "tqdm" "pandas" "safetensors"

In [ ]:
# ============================================================
# 2. Import + kiểm tra GPU
# ============================================================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import json
import math
import shutil
import gc
from pathlib import Path
from datetime import datetime

import torch
import pandas as pd
from tqdm.auto import tqdm

if not torch.cuda.is_available():
    raise RuntimeError(
        "Colab hiện tại KHÔNG có GPU/CUDA.\n"
        "QLoRA 4-bit với bitsandbytes bắt buộc cần GPU NVIDIA.\n"
        "Vào Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU/L4/A100.\n"
        "Sau đó Restart runtime và chạy lại từ đầu."
    )

free, total = torch.cuda.mem_get_info()
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)
print(f"Free VRAM before load: {free / 1024**3:.2f} GB / {total / 1024**3:.2f} GB")


In [ ]:
# ============================================================
# 3. Mount Drive + đọc dữ liệu
# Cấu trúc mong đợi:
# /content/drive/MyDrive/Final_Deeplearning/Split/train/train.jsonl
# /content/drive/MyDrive/Final_Deeplearning/Split/valid/valid.jsonl
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = Path('/content/drive/MyDrive/Final_Deeplearning/Split')
TRAIN_JSONL = DATA_ROOT / 'train' / 'train.jsonl'
VALID_JSONL = DATA_ROOT / 'valid' / 'valid.jsonl'

for path in [TRAIN_JSONL, VALID_JSONL]:
    assert path.exists(), f'Không tìm thấy file: {path}'


def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise ValueError(f'Lỗi JSONL dòng {line_no} ở {path}: {e}')
    return rows

train_rows = read_jsonl(TRAIN_JSONL)
valid_rows = read_jsonl(VALID_JSONL)

print('DATA_ROOT:', DATA_ROOT)
print('Train QA:', len(train_rows), 'Images:', len({x.get('image') for x in train_rows}))
print('Valid QA:', len(valid_rows), 'Images:', len({x.get('image') for x in valid_rows}))

assert len(train_rows) == 23211, f'Tập train hiện tại có {len(train_rows)} dòng, không phải 23211. Nếu dataset đã đổi thì sửa assert này.'
train_rows[:2]

In [ ]:
# ============================================================
# 4. CONFIG TRAIN NỐI TIẾP THEO CHUNK
# Chunk 1 train từ base model.
# Chunk 2 load adapter chunk 1 rồi train tiếp.
# Chunk 3 load adapter chunk 2 rồi train tiếp, ...
# ============================================================

# Notebook này đã sửa để CHUNK 2 lưu thẳng vào folder bạn đang mở trên Drive:
# /content/drive/MyDrive/Final_Deeplearning/qwen25_vl_herb_qlora_10chunks
# Sau khi chạy xong sẽ có:
#   chunk_02_02501_05000_work
#   chunk_02_02501_05000_adapter

# SỬA DUY NHẤT BIẾN NÀY CHO TỪNG NOTEBOOK
CHUNK_ID = 2

CHUNK_SIZE = 2500
MAX_VALID_SAMPLES = 500
RUN_EVAL_AFTER_CHUNK = False  # True nếu muốn evaluate sau mỗi chunk; False để tiết kiệm thời gian.

TRAIN_ROWS_ALL = train_rows
VALID_ROWS_ALL = valid_rows
TOTAL_TRAIN_ROWS = len(TRAIN_ROWS_ALL)

START_INDEX = (CHUNK_ID - 1) * CHUNK_SIZE
END_INDEX = min(CHUNK_ID * CHUNK_SIZE, TOTAL_TRAIN_ROWS)

train_rows_chunk = TRAIN_ROWS_ALL[START_INDEX:END_INDEX]
valid_rows_eval = VALID_ROWS_ALL[:MAX_VALID_SAMPLES] if MAX_VALID_SAMPLES else VALID_ROWS_ALL

assert TOTAL_TRAIN_ROWS == 23211, f"Tập train hiện tại có {TOTAL_TRAIN_ROWS} dòng, không phải 23211. Nếu dataset đổi thì sửa assert này."
assert 1 <= CHUNK_ID <= math.ceil(TOTAL_TRAIN_ROWS / CHUNK_SIZE), "CHUNK_ID không hợp lệ."
assert len(train_rows_chunk) > 0, "Chunk rỗng, kiểm tra lại CHUNK_ID hoặc CHUNK_SIZE."

# Folder CHÍNH: lưu chung chunk_01, chunk_02, ... trong cùng một thư mục Drive.
# Đây chính là folder bạn đang mở trong ảnh.
SEQ_OUTPUT_DIR = Path('/content/drive/MyDrive/Final_Deeplearning/qwen25_vl_herb_qlora_10chunks')
SEQ_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_NAME = f"chunk_{CHUNK_ID:02d}_{START_INDEX + 1:05d}_{END_INDEX:05d}"
WORK_DIR = SEQ_OUTPUT_DIR / f"{CHUNK_NAME}_work"
ADAPTER_OUTPUT_DIR = SEQ_OUTPUT_DIR / f"{CHUNK_NAME}_adapter"
SUMMARY_PATH = ADAPTER_OUTPUT_DIR / 'chunk_summary.json'

if CHUNK_ID == 1:
    PREV_ADAPTER_DIR = None
else:
    prev_start = (CHUNK_ID - 2) * CHUNK_SIZE
    prev_end = min((CHUNK_ID - 1) * CHUNK_SIZE, TOTAL_TRAIN_ROWS)
    PREV_CHUNK_NAME = f"chunk_{CHUNK_ID - 1:02d}_{prev_start + 1:05d}_{prev_end:05d}"
    PREV_ADAPTER_DIR = SEQ_OUTPUT_DIR / f"{PREV_CHUNK_NAME}_adapter"

print('TOTAL_TRAIN_ROWS:', TOTAL_TRAIN_ROWS)
print('CHUNK_ID:', CHUNK_ID)
print('CHUNK_NAME:', CHUNK_NAME)
print('Train samples 1-based:', START_INDEX + 1, '->', END_INDEX)
print('Num train samples:', len(train_rows_chunk))
print('Num valid eval samples:', len(valid_rows_eval))
print('OUTPUT ROOT:', SEQ_OUTPUT_DIR)
print('PREV_ADAPTER_DIR:', PREV_ADAPTER_DIR)
print('ADAPTER_OUTPUT_DIR:', ADAPTER_OUTPUT_DIR)
print('WORK_DIR:', WORK_DIR)

if CHUNK_ID > 1:
    expected_prev_adapter = (
        (PREV_ADAPTER_DIR / 'adapter_model.safetensors').exists()
        or (PREV_ADAPTER_DIR / 'adapter_model.bin').exists()
    )
    if not expected_prev_adapter:
        raise FileNotFoundError(
            'Không tìm thấy adapter chunk trước trong cùng folder Drive.\n'
            f'Cần có file adapter_model.safetensors hoặc adapter_model.bin tại:\n{PREV_ADAPTER_DIR}\n'
            'Với chunk 2, hãy chắc chắn folder chunk_01_00001_02500_adapter đã chạy xong và nằm trong qwen25_vl_herb_qlora_10chunks.'
        )


In [ ]:
# ============================================================
# 5. Load Qwen2.5-VL 3B 4-bit + tạo/load LoRA adapter
# - CHUNK_ID = 1: tạo adapter mới từ base model.
# - CHUNK_ID > 1: load adapter chunk trước với is_trainable=True rồi train tiếp.
# ============================================================

import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from qwen_vl_utils import process_vision_info

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

# Giảm vision tokens để tránh OOM trên T4.
MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = 256 * 28 * 28

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

model.config.use_cache = False

# Không gọi prepare_model_for_kbit_training vì trên T4 dễ OOM.
# Freeze base model; chỉ LoRA adapter được train.
for param in model.parameters():
    param.requires_grad = False

if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

if PREV_ADAPTER_DIR is None:
    print('Chunk 1: tạo LoRA adapter mới từ base model.')

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=[
            'q_proj', 'k_proj', 'v_proj', 'o_proj',
            'gate_proj', 'up_proj', 'down_proj',
        ],
    )
    model = get_peft_model(model, lora_config)
else:
    print('Load adapter chunk trước để train tiếp:')
    print(PREV_ADAPTER_DIR)

    if not (PREV_ADAPTER_DIR / 'adapter_model.safetensors').exists() and not (PREV_ADAPTER_DIR / 'adapter_model.bin').exists():
        raise FileNotFoundError(
            f'Không tìm thấy adapter_model trong folder chunk trước:\n{PREV_ADAPTER_DIR}\n'
            'Phải chạy xong chunk trước rồi mới chạy chunk hiện tại.'
        )

    model = PeftModel.from_pretrained(
        model,
        str(PREV_ADAPTER_DIR),
        is_trainable=True,
    )

model.print_trainable_parameters()

free, total = torch.cuda.mem_get_info()
print(f'Free VRAM after load: {free / 1024**3:.2f} GB / {total / 1024**3:.2f} GB')


In [ ]:
# ============================================================
# 6. Prompt, resolve đường dẫn ảnh, Dataset, collator
# ============================================================
from torch.utils.data import Dataset


def image_root_for_split(split):
    return DATA_ROOT / split


def normalize_image_relpath(x):
    return str(x).strip().replace('\\', '/').lstrip('/')


def resolve_image_path(row, split=None):
    raw = normalize_image_relpath(row['image'])
    p = Path(raw)
    candidates = []

    if p.is_absolute():
        candidates.append(p)

    split_candidates = []
    if split:
        split_candidates.append(split)

    for s in ['train', 'valid', 'val', 'test']:
        if s not in split_candidates:
            split_candidates.append(s)

    for s in split_candidates:
        split_root = image_root_for_split(s)
        candidates.append(split_root / raw)                 # images/Pxxxx.png
        candidates.append(split_root / 'images' / p.name)   # Pxxxx.png

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        'Không tìm thấy ảnh.\n'
        f"id: {row.get('id')}\n"
        f"split truyền vào: {split}\n"
        f"image trong JSONL: {row.get('image')}\n"
        'Đã thử:\n' + '\n'.join(str(x) for x in candidates[:20])
    )


def build_user_text(question):
    return (
        'Bạn là hệ thống hỏi đáp ảnh dược liệu Việt Nam. '
        'Hãy trả lời câu hỏi bằng tiếng Việt, thật ngắn gọn, tối đa 10 từ. '
        'Không giải thích thêm.\n'
        f'Câu hỏi: {question}'
    )


def make_messages(row, include_answer=True):
    split = row.get('__split', None)
    image_path = resolve_image_path(row, split=split)

    messages = [
        {
            'role': 'user',
            'content': [
                {
                    'type': 'image',
                    'image': str(image_path),
                    'min_pixels': MIN_PIXELS,
                    'max_pixels': MAX_PIXELS,
                },
                {'type': 'text', 'text': build_user_text(row['question'])},
            ],
        }
    ]

    if include_answer:
        messages.append({
            'role': 'assistant',
            'content': [{'type': 'text', 'text': str(row['answer'])}],
        })

    return messages


class HerbVQADataset(Dataset):
    def __init__(self, rows, split):
        self.rows = rows
        self.split = split

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = dict(self.rows[idx])
        row['__split'] = self.split
        return row


def collate_fn(batch):
    full_messages = [make_messages(row, include_answer=True) for row in batch]
    prompt_messages = [make_messages(row, include_answer=False) for row in batch]

    full_texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in full_messages
    ]
    prompt_texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in prompt_messages
    ]

    image_inputs, video_inputs = process_vision_info(full_messages)

    inputs = processor(
        text=full_texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    )

    labels = inputs['input_ids'].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    prompt_tokenized = processor.tokenizer(prompt_texts, add_special_tokens=False)
    for i, prompt_ids in enumerate(prompt_tokenized['input_ids']):
        prompt_len = len(prompt_ids)
        labels[i, :prompt_len] = -100

    inputs['labels'] = labels
    return inputs


def check_missing_images(dataset, max_show=10):
    missing = []
    for i in range(len(dataset)):
        row = dataset[i]
        split = row.get('__split', getattr(dataset, 'split', None))
        try:
            _ = resolve_image_path(row, split=split)
        except Exception as e:
            missing.append((i, row.get('id'), split, row.get('image'), str(e)))

    print('Tổng samples:', len(dataset))
    print('Số ảnh lỗi:', len(missing))
    for item in missing[:max_show]:
        print(item)
    return missing


train_dataset = HerbVQADataset(train_rows_chunk, 'train')
valid_dataset = HerbVQADataset(valid_rows_eval, 'valid')

missing_train = check_missing_images(train_dataset)
missing_valid = check_missing_images(valid_dataset)

if len(missing_train) > 0 or len(missing_valid) > 0:
    raise FileNotFoundError('Vẫn còn ảnh lỗi đường dẫn. Xem danh sách ở trên.')


In [ ]:
# ============================================================
# 7. Train chunk nối tiếp + lưu adapter
# ============================================================
import gc
import json
import torch
from datetime import datetime
from transformers import TrainingArguments, Trainer


def find_last_checkpoint(output_dir):
    output_dir = Path(output_dir)
    if not output_dir.exists():
        return None

    checkpoints = []
    for p in output_dir.glob('checkpoint-*'):
        if p.is_dir():
            try:
                step = int(p.name.split('-')[-1])
                checkpoints.append((step, p))
            except Exception:
                pass

    if not checkpoints:
        return None

    return str(sorted(checkpoints, key=lambda x: x[0])[-1][1])


adapter_exists = (
    (ADAPTER_OUTPUT_DIR / 'adapter_model.safetensors').exists()
    or (ADAPTER_OUTPUT_DIR / 'adapter_model.bin').exists()
)
summary_exists = SUMMARY_PATH.exists()

if adapter_exists and summary_exists:
    print('Adapter chunk này đã tồn tại và có chunk_summary.json, bỏ qua train lại:')
    print(ADAPTER_OUTPUT_DIR)

elif adapter_exists and not summary_exists:
    raise RuntimeError(
        'Adapter đã tồn tại nhưng thiếu chunk_summary.json.\n'
        'Để tránh sai trạng thái nối tiếp, hãy xóa folder adapter này rồi chạy lại chunk hiện tại:\n'
        f'{ADAPTER_OUTPUT_DIR}'
    )

else:
    training_args = TrainingArguments(
        output_dir=str(WORK_DIR),
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=1,
        logging_steps=20,

        # Có checkpoint để resume nếu Colab bị ngắt giữa chunk.
        eval_strategy='no',
        save_strategy='steps',
        save_steps=300,
        save_total_limit=1,

        fp16=True,
        bf16=False,

        remove_unused_columns=False,
        report_to='none',

        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},

        optim='paged_adamw_8bit',
        warmup_ratio=0.03,
        label_names=['labels'],

        dataloader_num_workers=0,
        dataloader_pin_memory=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset if RUN_EVAL_AFTER_CHUNK else None,
        data_collator=collate_fn,
    )

    resume_ckpt = find_last_checkpoint(WORK_DIR)
    print('Resume checkpoint:', resume_ckpt)

    if resume_ckpt:
        train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
    else:
        train_result = trainer.train()

    ADAPTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(ADAPTER_OUTPUT_DIR))
    processor.save_pretrained(str(ADAPTER_OUTPUT_DIR))

    eval_metrics = {}
    eval_loss = None
    if RUN_EVAL_AFTER_CHUNK:
        print('\n[EVAL] Đánh giá chunk trên valid subset')
        eval_metrics = trainer.evaluate()
        eval_loss = float(eval_metrics.get('eval_loss', 999999.0))

    summary = {
        'train_mode': 'sequential_chunk_training',
        'chunk_id': CHUNK_ID,
        'chunk_name': CHUNK_NAME,
        'prev_adapter_dir': str(PREV_ADAPTER_DIR) if PREV_ADAPTER_DIR is not None else None,
        'train_start_index_0_based': START_INDEX,
        'train_end_index_0_based_exclusive': END_INDEX,
        'train_sample_from_1_based': START_INDEX + 1,
        'train_sample_to_1_based': END_INDEX,
        'num_train_samples': len(train_rows_chunk),
        'num_valid_samples': len(valid_rows_eval) if RUN_EVAL_AFTER_CHUNK else 0,
        'eval_loss': eval_loss,
        'eval_metrics': eval_metrics,
        'adapter_dir': str(ADAPTER_OUTPUT_DIR),
        'work_dir': str(WORK_DIR),
        'model_id': MODEL_ID,
        'finished_at': datetime.now().isoformat(),
    }

    with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print('\nSaved adapter:', ADAPTER_OUTPUT_DIR)
    print('Saved summary:', SUMMARY_PATH)
    print('Eval loss:', eval_loss)

    # Nếu chạy đến chunk cuối thì copy ra final_adapter.
    if END_INDEX == TOTAL_TRAIN_ROWS:
        FINAL_ADAPTER_DIR = SEQ_OUTPUT_DIR / 'final_adapter'
        if FINAL_ADAPTER_DIR.exists():
            shutil.rmtree(FINAL_ADAPTER_DIR)
        shutil.copytree(ADAPTER_OUTPUT_DIR, FINAL_ADAPTER_DIR)
        print('\nFINAL ADAPTER:', FINAL_ADAPTER_DIR)

    del trainer
    gc.collect()
    torch.cuda.empty_cache()
